# Spatial Data Science with an LoD2 city model

The purpose of this Notebook is to illustrate a CityJSON-based spatial data science workflow with a LoD2 city model.

In [1]:
#load the magic

%matplotlib inline
import os
import sys
from pathlib import Path
import webbrowser

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, shape, mapping
import json
import geojson

from osgeo import gdal, osr

import matplotlib.pyplot as plt

In [2]:
import LoD2city3D

from cjio import cityjson

In [3]:
jparams = json.load(open('./param/uEstate_param.json'))   

In [4]:
# With the new 0.10.x syntax:
with open(jparams['cjsn_out'], 'r') as f:
    cm = cityjson.reader(f)

In [5]:
print(cm)

CityJSON version = 2.0.0
EPSG = 32734
bbox = [ 263778.110 6241078.022 28.800 265430.307 6242637.108 193.800 ]
Extensions = ['Energy']
=== CityObjects ===
|-- TINRelief (1)
|-- Road (58)
|-- Building (308)
    |-- BuildingInstallation (145)
        |-- +Energy-PVSystem (145)
|-- LandUse (3)
|-- SolitaryVegetationObject (11)
materials = False
textures = False


In [6]:
#- harvest the crs
theinfo = cm.get_info()
crs = theinfo[1]
print(crs)

EPSG = 32734


In [7]:
# 1. Grab the objects directly from the reader
buildings = {
    obj_id: obj 
    for obj_id, obj in cm.j['CityObjects'].items() 
    if obj.get('type') == 'Building'
}
vertices = cm.j['vertices']

In [8]:
transform_meta = cm.j.get('transform', {})
transform_meta

{'scale': [0.001, 0.001, 0.001],
 'translate': [264604.208314, 6241857.564832, 28.8]}

In [9]:
# 2. Extract surfaces with the transformation metadata included
geojson_output = LoD2city3D.extract_lod_surfaces(
    buildings, 
    cm.j['vertices'], 
    transform_meta=transform_meta, 
    crs_name=crs[7:]#"EPSG:32734"
)

# 3. Instantiate your clean GeoDataFrameLite directly from the output collection payload
gdf = LoD2city3D.GeoDataFrameLite.from_json(geojson_output)
gdf.crs = crs[7:]#"EPSG:32734"

In [10]:
#gdf.loc[1914].geometry

In [11]:
gdf.lod.unique()

array(['1.0', '2.0'], dtype=object)

In [12]:
gdf.tail(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
3427,osm_1469054027,1.0,GenericSurface,solid,185.659302,0.0,0.0,85.7,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264879.473, 6241...","POLYGON Z ((264873.028314 6241703.458832 85.7,..."
3428,osm_1469054027,1.0,GenericSurface,solid,185.659302,0.0,0.0,78.8,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264879.473, 6241...","POLYGON Z ((264865.929314 6241707.344832 78.8,..."


In [13]:
unique_building_count = gdf['building_id'].nunique()
print(f"Total number of unique buildings: {unique_building_count}")

Total number of unique buildings: 308


<div class="alert alert-block alert-info"><b>Craveat</b> 
    
This community *(Woodstock, Salt River and Observatory)* is the second oldest community in South Africa. The buildings are old. Many have been repurposed. To account for refurbishment *--be as representative as possible--* and conform to the **[OpenStreetMap Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide)** we typically tag these:  
`building=*` *~ the original purpose* `+` `building:use=*` *~ the current use*.

Furthermore; tagging in this community identifies **social housing, social facilities** (care home, shelter, etc.) **and informal housing** (backyard dwelling, shack, etc.) as `building / :use=residential`. **Student accomodation** includes the `residential=student` tag
</div>

In [14]:
# --- Account for idiosyncratic mapping at the surface level ---
gdf2 = gdf.copy()

# 1. Update surface classification if a current 'building:use' is present
if 'attr_building:use' in gdf2.columns:
    # Locate active residential updates while ignoring NaN records safely
    use_mask = (gdf2['attr_building:use'] == 'residential') & gdf2['attr_building:use'].notna()
    gdf2.loc[use_mask, 'attr_building'] = gdf2.loc[use_mask, 'attr_building:use']

# 2. Update surface classification to show active student housing usage
if 'attr_residential' in gdf2.columns:
    # Locate explicit student housing tags
    student_mask = (gdf2['attr_residential'] == 'student') & gdf2['attr_residential'].notna()
    gdf2.loc[student_mask, 'attr_building'] = gdf2.loc[student_mask, 'attr_residential']

# 1. Deduplicate by building_id so each physical asset is only counted once
unique_buildings_df = gdf2.drop_duplicates(subset=['building_id'])

# --- Verify the updated semantic surface distribution across the community ---
print("Updated building attribute distribution across harvested surfaces:")
print(unique_buildings_df['attr_building'].value_counts())

Updated building attribute distribution across harvested surfaces:
attr_building
house        295
garage         7
yes            4
warehouse      1
office         1
Name: count, dtype: int64


## 1. Spatial Data Science

<div class="alert alert-block alert-warning"><b>We start with basic spatial analysis</b>  
    
     
- We'll [estimate the population](#Section1a), within our area of interest, and then  
- calculate the [Building Volume Per Capita (BVPC)](#Section1b).
</div>

While estimating population is well documented; recent investigations to **understand overcrowding** have led to newer measurements.  

The most noteable of these is **Building Volume Per Capita (BVPC)** [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs). BVPC is the cubic meters of building per person. **BVPC tells us how much space one person has per residential living unit** (a house / apartment / etc.). It is ***a proxy measure of economic inequality and a direct measure of housing inequality***.

BVPC builds on the work of [(Reddy, A and Leslie, T.F., 2013)](https://www.tandfonline.com/doi/abs/10.1080/02723638.2015.1060696?journalCode=rurb20) and attempts to integrate with several **[Sustainable Development Goals](https://sdgs.un.org/goals)** (most noteably: **[SDG 11: Developing sustainable cities and communities](https://sdgs.un.org/goals/goal11)**) and captures the average ***'living space'*** each person has in their home.

<div class="alert alert-block alert-info"><b>These analysis expect the user to have some basic knowledge about the environment under inquiry / investigation</b> </div>

In [15]:
len(unique_buildings_df)

308

In [16]:
unique_buildings_df['attr_building'].unique()

array(['warehouse', 'house', 'garage', 'yes', 'office'], dtype=object)

<div class="alert alert-block alert-success"><b>1.  a) Estimate Population:</b> 
    
_(with population growth rate and population projection possible too)_ </div>

In [17]:
#- some data wrangling
with pd.option_context("future.no_silent_downcasting", True):
    unique_buildings_df = unique_buildings_df.assign(**{
        col: pd.to_numeric(
            unique_buildings_df[col].fillna(0).infer_objects(copy=False), errors='coerce'
        )
        for col in ['attr_building:flats', 'attr_building:units', 'attr_beds', 'attr_rooms', 'attr_building:levels']
        if col in unique_buildings_df.columns
    })

unique_buildings_df.head(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
0,osm_13705114,1.0,GenericSurface,solid,536.217387,90.0,204.054416,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264304.860314 6241972.752832 69.60...
12,osm_739615941,2.0,RoofSurface,solid,6.031372,0.0,0.000000,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,house,2.0,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264276.174, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...


**This area is urban with single and 2-storey level housing units. To estimate population is thus pretty straight forward.**

<div class="alert alert-block alert-info"><b>We start with local knowledge.</b></div>

**On average there are roughly `4` people per `building:house` in this area.**  

**[social housing](https://en.wikipedia.org/wiki/Public_housing)** is tagged `building:residential` with the `3` people per building or `building:flats * 3` if the building is an *apartment-type* complex

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    
We will execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [18]:
#- average number of residents per formal house
f_house = 4
#- average number of residents per informal structure
inf_structure = 3

<div class="alert alert-block alert-warning"><b></b>  
    
**Furthermore:**    
- [social housing](https://en.wikipedia.org/wiki/Public_housing) is tagged `building:residential` with the number of occupants iether the number of informal structure occupants or `building:flats * inf_structure`
- A `social_facility` (carehome, shelter, etc.) harvests the beds `key:value` pair.
- `building:apartments` harvests the `building:flats` `key:value` pair (the number of units) to calculate `*3` people per  
    - ***Student accomodation***:  
>    - University owed: is tagged `building:dormitory` with `residential:university` and harvests the `beds` or `rooms` *'key:value'* pair.
>    - Private for-profit: is tagged `building:residential` or `:dormitory` with `residential:student` and then harvests the `building:flats` or `:rooms` *'key:value'* pair *(the number of units)* to calculate `*1` people per apartment; if `level: > 1` else `*3` people in a house share.  

**The tagging scheme and numbers is based on *how your community is mapped* and local knowledge**
</div>

In [19]:
#- population estimation
gdf_pop = unique_buildings_df.copy()
c = gdf_pop.columns

def pop(row):
    #- formal house
    if row['attr_building'] == 'house' or row['attr_building'] == 'semidetached_house':
        return f_house
    if row['attr_building'] == 'terrace' or row['attr_building'] == 'terraced':
        if 'attr_building:units' in c and row['attr_building:units'] != 0:
            return row['attr_building:units'] * f_house
        else:
            f_house

    #- informal structure (shack)
    if row['attr_building'] == 'cabin':
        return inf_structure
        
    #- in this case social housing
    if row['attr_building'] == 'residential' and 'social_facility' in c and row['attr_social_facility'] is np.nan:
        if row['attr_building:levels'] > 1:
            if 'attr_rooms' in c and row['attr_rooms'] != 0:
                return row['attr_rooms']
            if 'attr_building:flats' in c and row['attr_building:flats'] != 0:
                return row['attr_building:flats'] * inf_structure
        else:
            return inf_structure
    #-- social facility [shelter / carehome]
    if row['attr_building'] == 'residential' and row['attr_social_facility'] is not np.nan:
        if 'attr_building:units' in c and row['attr_building:units'] != 0:
            return row['attr_building:units'] * inf_structure
        else: 
            return row['attr_beds']
                
    #- formal apartment
    if row['attr_building'] == 'apartments':
        return row['attr_building:flats'] * 3
        
    #- private student residence 
    if row['attr_building'] == 'student':
        if row['attr_building:levels'] > 1:
            return row['attr_building:flats']
        else:
            return 3
    # university owned student residence
    if row['attr_building'] == 'dormitory' and row['attr_residential'] == 'university':
        if row['attr_building:levels'] > 1:
            if 'attr_rooms' in c and row['attr_rooms'] != 0:
                return row['rooms']
            if 'attr_beds' in c and row['attr_beds'] != 0:
                return row['attr_beds']
        else:
            return 3

gdf_pop['pop'] = gdf_pop.apply(lambda x: pop(x), axis=1)

est_pop = int(gdf_pop['pop'].sum())
print('The estimated population is:', est_pop)

The estimated population is: 1180


**[Statistics South Africa (STATSA)](https://www.statssa.gov.za) does not typically release official statistics at a neighbourhood level but its possible to find dedicated and resourceful geospatial professionals, in your immediate environment, who are just *plodding-away* doing awesome and useful stuff.  
These numbers are based on disaggregated [Statistics South Africa (STATSA)](https://www.statssa.gov.za) [Census 2011](https://www.statssa.gov.za/?page_id=3839) and can be accessed [here](https://census2011.adrianfrith.com/place/199041033)** 

**University Estate 987** (6 577: salt river, 9 207: observatory and 12 656: woodstock ). 

We can calculate the annual population growth rate using the formula for **[Annual population growth](https://databank.worldbank.org/metadataglossary/health-nutrition-and-population-statistics/series/SP.POP.GROW):**

$$r = \frac{\ln{[\frac{End Population}{Start Population}}]}{n} * 100 = \frac{\ln{[\frac{1 180}{987}}]}{12} * 100   = 1.49\%$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [20]:
#- previous population
start_population = 987

#- period in years from the previous census
years = 12

In [21]:
#-execute
r = (np.log(est_pop/start_population)/years) * 100
print('population growth rate of approximately:', round(r, 2), '%')

population growth rate of approximately: 1.49 %


To conclude; we can project into the future with a very basic formula to estimate the population _x_-years from now:  

$$p  = P_o * (1 + r)^{t} = p = 1180 * (1 + 0.0149)^{10}  = 1 367$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the variables in the _`cell`_ below** </div>

In [22]:
#- period in years from now
years = 10

In [23]:
#- account for non-residential areas without failure
#- helper function
def safe_population_estimate(est_pop, r, years):
    try:
        p = est_pop * (1 + (r / 100))**years
        return int(p)
    except Exception as e:
        print(f"Population estimate failed: {e}")
        return None  # keeps notebook running

#- execute function
p = safe_population_estimate(est_pop, r, years)

#- shows error and moves on
if p is not None:
    print(f"estimated population {years} years from now: {p}")

estimated population 10 years from now: 1367


<div class="alert alert-block alert-success"><b>1. b) Building Volume Per Capita (BVPC):</b>  
BVPC = total population of a community divided by sum of building volume</div>

In [24]:
#- the surfaces
gdf.head(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
0,osm_13705114,1.0,GenericSurface,solid,536.217387,90.0,204.054416,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264304.860314 6241972.752832 69.60...
1,osm_13705114,1.0,GenericSurface,solid,201.707554,90.0,114.281097,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264375.82431399997 6241941.076832 ...


In [25]:
def calculate_facet_signed_volume(geometry):
    """
    Computes the signed volume contribution of an individual 3D face polygon 
    relative to the origin (0,0,0). The sum of all faces yields a closed shell volume.
    """
    # Extract the true [X, Y, Z] vertex coordinates preserved by your extraction pipeline
    # Fail-safe check for missing or invalid geometry records
    if geometry is None or hasattr(geometry, 'is_empty') and geometry.is_empty:
        return 0.0
        
    # Extract the true 3D coordinate ring [X, Y, Z] directly from the Shapely object
    vertices_3d = np.array(geometry.exterior.coords)
    #vertices_3d = np.array(geometry_dict['coordinates'][0])
    
    # A valid closed ring must have at least an initial triangle + closure point (4 pts)
    if len(vertices_3d) < 4:
        return 0.0
        
    face_volume = 0.0
    p0 = vertices_3d[0]
    
    # Step through the 3D vertex ring to isolate individual triangles relative to the origin
    for i in range(1, len(vertices_3d) - 2):
        p1 = vertices_3d[i]
        p2 = vertices_3d[i + 1]
        
        # The triple scalar product np.dot(p0, np.cross(p1, p2)) / 6.0 
        # yields the exact signed volume of the tetrahedron formed with the origin
        face_volume += np.dot(p0, np.cross(p1, p2)) / 6.0
        
    return face_volume

#print("Calculating signed volume contributions for all individual 3D facets...")
gdf['face_signed_volume'] = gdf['geometry'].apply(calculate_facet_signed_volume)

# Group the surfaces by their parent building, sum the signed elements, and take the absolute total
building_volume_series = gdf.groupby('building_id')['face_signed_volume'].sum().abs()
volume_lookup_df = building_volume_series.reset_index(name='volume_m3')
gdf.head(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,face_signed_volume
0,osm_13705114,1.0,GenericSurface,solid,536.217387,90.0,204.054416,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264304.860314 6241972.752832 69.60...,-1.038053e+09
1,osm_13705114,1.0,GenericSurface,solid,201.707554,90.0,114.281097,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264375.82431399997 6241941.076832 ...,-1.563760e+08


In [27]:
# ==============================================================================
# STEP 2: Deduplicate to isolate the unique buildings list
# ==============================================================================
unique_buildings = gdf.drop_duplicates(subset=['building_id']).copy()

# Remove face-specific columns to keep the asset table clean
surface_cols = ['geometry', 'surface_type', 'tilt_deg', 'aspect_deg', 'mean_z_absolute', 'face_signed_volume']
unique_buildings = unique_buildings.drop(columns=[col for col in surface_cols if col in unique_buildings.columns])

# Apply the aggregated volume data via the merge
unique_buildings = unique_buildings.merge(volume_lookup_df, on='building_id', how='left')
#print(f"Total number of unique buildings: {unique_buildings['building_id'].nunique()}")

In [28]:
#- remove the volume of the ground floor (unoccupied) when building:levels > 7 [this is an arbitrary number based on local knowledge]
#- typically the space is reserved for some other function: retail, etc. 
GROUND_FLOOR_HEIGHT_M = 2.8
adjusted_volumes = []

for _, row in unique_buildings.iterrows():
    current_vol = row['volume_m3']
    
    # Safely handle level parsing
    levels = row.get('attr_building:levels', 0)
    levels = float(levels) if pd.notna(levels) and str(levels).strip() != '' else 0
    
    has_no_social_facility = ('attr_social_facility' not in unique_buildings.columns or 
                              pd.isna(row.get('attr_social_facility')))
    
    is_residential_target = str(row.get('attr_building')).lower() in ['residential', 'apartments', 'student']
    
    # Deduct the 1st floor volume if the residential asset is > 7 stories
    if is_residential_target and levels > 7 and has_no_social_facility:
        footprint_area = row.get('3d_area', 0.0)
        deducted_vol = current_vol - (footprint_area * GROUND_FLOOR_HEIGHT_M)
        adjusted_volumes.append(max(0.0, deducted_vol))
    else:
        adjusted_volumes.append(current_vol)

# Overwrite the merged values with your refined domestic metrics
gdf_pop['volume_m3'] = adjusted_volumes
gdf_pop.head(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,...,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,pop,volume_m3
0,osm_13705114,1.0,GenericSurface,solid,536.217387,90.0,204.054416,72.36,13705114,197 Nelson Mandela Boulevard Cape Town,...,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...",POLYGON Z ((264304.860314 6241972.752832 69.60...,NaN,16778.099256
12,osm_739615941,2.0,RoofSurface,solid,6.031372,0.0,0.000000,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,...,2.0,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264276.174, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,4.0,1013.307644


In [29]:
gdf_pop['bvpc'] = np.where(
    gdf_pop['pop'] > 0,
    gdf_pop['volume_m3'] / gdf_pop['pop'],
    np.nan
)

gdf_pop.tail(2)

,building_id,lod,surface_type,geometry_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,...,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,pop,volume_m3,bvpc
3413,osm_1459039139,1.0,GenericSurface,solid,25.007454,90.0,298.602963,93.54,1459039139,NaN,...,NaN,4.1,96.3,91.9,4FRW3F62+9H4,"[[[264446.096, 6241675.189], [264443.176, 6241...",POLYGON Z ((264446.09631399997 6241675.188832 ...,NaN,245.504498,NaN
3419,osm_1469054027,1.0,GenericSurface,solid,91.914369,90.0,118.935540,81.56,1469054027,66 Selbourne Road University Estate Cape Town,...,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264879.473, 6241...","POLYGON Z ((264873.028314 6241703.458832 78.8,...",4.0,1281.049377,320.262344


In [30]:
print(gdf_pop['bvpc'].describe())

count    295.000000
mean     219.383648
std      127.966001
min       34.487385
25%      133.022154
50%      187.667593
75%      271.497537
max      971.828275
Name: bvpc, dtype: float64


In [31]:
bvpc = round(gdf_pop['volume_m3'].sum() / est_pop, 3)

print('Building Volume Per Capita (BVPC):', bvpc)

Building Volume Per Capita (BVPC): 248.669


<div class="alert alert-block alert-info"><b></b>

**This BVPC value is for all buildings with a `population > 0`. Buildings people live in** *(homes)*. 

And we can seperate `building:house` from `building:cabin` and `building:residential` to undertand the differences between ***formal and informal*** housing in this area.
    
**We want to understand the living space *(the cubic-meter BVPC value)* each person has in thier home**
</div>

In [32]:
formal = gdf_pop[gdf_pop["attr_building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments'])].copy()
f_pop = int(formal['pop'].sum())

informal = gdf_pop[gdf_pop["attr_building"].isin(['residential', 'cabin'])].copy()
inf_pop = int(informal['pop'].sum())

#- student
stu = gdf_pop[gdf_pop["attr_building"].isin(['student', 'dormitory'])].copy()
stu_pop = int(stu['pop'].sum())

bvpc_formal = round(formal['volume_m3'].sum() / formal['pop'].sum(), 3)
bvpc_informal = round(informal['volume_m3'].sum() / informal['pop'].sum() if informal['pop'].sum() != 0 else 0, 3)
bvpc_stu = round(stu['volume_m3'].sum() / stu['pop'].sum() if stu['pop'].sum() != 0 else 0, 3)

print('FORMAL: Population: ', f_pop, ' with Building Volume Per Capita (BVPC):', bvpc_formal)
print('')
print('STUDENT RESIDENCE: Population: ', stu_pop, ' with Building Volume Per Capita (BVPC):', bvpc_stu)
print('')
print('INFORMAL: Population: ', inf_pop, ' with Building Volume Per Capita (BVPC)', bvpc_informal)

FORMAL: Population:  1180  with Building Volume Per Capita (BVPC): 219.384

STUDENT RESIDENCE: Population:  0  with Building Volume Per Capita (BVPC): 0

INFORMAL: Population:  0  with Building Volume Per Capita (BVPC) 0


## 2. Further examples of Spatial Data Science *(renewable energy)*:

<div class="alert alert-block alert-warning"><b></b>

**Let's attempt to understand the % of homes and population served with renewable energy.**
</div>
    
[**SDG**](https://sdgs.un.org/goals) indicators are typically calculated at **region and national scales**.  
Here, because we are working with highly detailed, local data, we can explore what a [**Tier 3 local indicator**](https://unstats.un.org/sdgs/metadata/) might look like at a ***neighbourhood level***.
<br>

In this section 3. we evaluate [**SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all**](https://sdgs.un.org/goals/goal7) at a community level and estimate the **proportion of residential units and population** that have **direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH)*.

- Percentage of **households** served by rooftop renewable energy  
- Percentage of the **population** served by rooftop renewable energy
- And then we go even further to estimate the **Annual Solar Potential in MWh** *(theoretical maximum electricity)* that homes can harvest from the sun over the course of one year.
<div class="alert alert-block alert-info"><b></b> 
    
**PLEASE SUPPLY YOUR OWN [osm.pbf](https://wiki.openstreetmap.org/wiki/PBF_Format).**  

**Either crop an area directly from [OpenStreetMap]() with the ***[official tool](https://www.openstreetmap.org/export#map=3/0.70/22.15)***, select a predefined area [from any number of providers](https://wiki.openstreetmap.org/wiki/Planet.osm), such as ***[Geofabrik](https://download.geofabrik.de)***, or...**
</div>
<div class="alert alert-block alert-success"><b></b>
    
**... download your own. Provincial extracts for South Africa are available here:** *http://download.openstreetmap.fr/extracts/africa/south_africa/*</div>
</div>